# Lesson 18: Chat Interface — Your Main Workspace

## From Architecture to Conversation

Last lesson traced how everything connects internally. Now the **user-facing side** — the chat interface.

The **chat interface** is the **primary way to use this product**. Run `python output/chat.py` and talk naturally:

- *"Create an article about on-page SEO"*
- *"Is that article done yet?"*
- *"Retry article 3"*
- *"Load topics from my-topics.csv"*
- *"Show me all articles that are pending review"*

The AI understands your intent and acts — no need to memorize command syntax.

## Agno Team — A Specialized AI Team

The chat interface is built with **Agno Team** — a group of AI agents, each with a specific role.

How it works:
1. You send a message
2. The **Leader** (team manager) reads your request
3. The Leader **delegates** to the right team member
4. The team member does the work (calls tools) and returns results
5. The Leader **summarizes** and responds to you

Same pattern as a **project manager** — you say what you want, the PM assigns it.

In [ ]:
print("""
SEO Workspace Team:
  
  Leader (Claude Opus) — Receives requests, delegates to members
    |
    +-- Content Creator (Claude Sonnet)
    |     Tools: create_article, create_article_batch,
    |            retry_article, load_articles_from_csv
    |     Role: Create new articles, retry failures, import from CSV
    |
    +-- Status Tracker (Claude Sonnet)
    |     Tools: list_all_articles, get_article_details, 
    |            get_article_content, get_version_history
    |     Role: Check article status
    |
    +-- SEO Manager (Claude Sonnet)
          Tools: check_rankings,
                 set_article_published_url, get_ranking_history
          Role: Published URLs, rank tracking

Example conversation:
  You: "Create an article about on-page SEO"
  -> Leader delegates to Content Creator
  -> Content Creator runs the pipeline
  -> Leader: "Created article recABC123, 2000 words, file: content/seo-on-page.md"

  You: "Set the URL to https://blog.example.com/on-page-seo"
  -> Leader delegates to SEO Manager
  -> SEO Manager sets published_url in Airtable

  You: "Check the rankings for that article"
  -> Leader delegates to SEO Manager
  -> SEO Manager calls DataForSEO and returns positions
""")

## Workspace Tools

Team members call **workspace tools** to do their work. These are plain Python functions in `tools/workspace.py`:

| Tool | Member | Purpose |
|------|--------|--------|
| `create_article` | Content Creator | Create 1 article (runs pipeline) |
| `create_article_batch` | Content Creator | Create multiple articles at once |
| `retry_article` | Content Creator | Retry a failed article |
| `load_articles_from_csv` | Content Creator | Import topics from CSV |
| `list_all_articles` | Status Tracker | View all articles |
| `get_article_details` | Status Tracker | View details of 1 article |
| `get_article_content` | Status Tracker | View article content |
| `get_version_history` | Status Tracker | View version history |
| `check_rankings` | SEO Manager | Check SERP positions via DataForSEO |
| `set_article_published_url` | SEO Manager | Set live URL for an article |
| `get_ranking_history` | SEO Manager | View ranking data over time |

Same functions you learned in previous lessons — wrapped so team members can use them as tools.

## Quick Note: json.loads() and json.dumps()

You'll see these two functions when working with JSON data in Python:

- `json.loads(text)` — **load string**: converts a JSON text string into a Python dict/list
- `json.dumps(data)` — **dump string**: converts a Python dict/list into a JSON text string

```python
import json

# JSON string -> Python dict
text = '{"title": "SEO Guide", "words": 1500}'
data = json.loads(text)
print(data["title"])  # "SEO Guide"

# Python dict -> JSON string
back_to_text = json.dumps(data)
print(back_to_text)   # '{"title": "SEO Guide", "words": 1500}'
```

The "s" in loads/dumps stands for "string". There's also `json.load(file)` / `json.dump(data, file)` for reading/writing files directly.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

from tools.workspace import (
    create_article,
    create_article_batch,
    retry_article,
    load_articles_from_csv,
    list_all_articles,
    get_article_details,
    get_article_content,
    get_version_history,
    check_rankings,
    set_article_published_url,
    get_ranking_history,
)

# These are plain Python functions!
# Team members call them as tools

# Example: view all articles
import json
result = list_all_articles()
articles = json.loads(result)
print(f"Found {len(articles)} articles in workspace")
for a in articles:
    print(f"  {a['id']}: {a['topic']} ({a['status']})")

In [ ]:
# Example: SEO Manager tools (no API cost)
# These are the same functions the SEO Manager agent calls

# Check ranking history (reads from Airtable)
result = get_ranking_history("recABC123")  # Use a real article ID from above
print("Ranking history:")
print(result)

## Running the Chat Interface

Start the chat from the project root:

```bash
python output/chat.py
```

You'll see:
- A welcome message and instructions
- A `>` prompt for your messages
- AI responses and actions
- Type `quit` or `exit` to leave

Chat history is saved in `chat_sessions.db`, so you can continue from where you left off.

In [ ]:
print("To run the chat interface:")
print()
print("  python output/chat.py")
print()
print("Chat history is saved in chat_sessions.db")
print("so you can continue from where you left off.")

## Reading the Production Code

You've seen every concept in action through the notebooks. The production code in `output/` uses **the same patterns** — here's where to find each piece:

### Mapping: Lesson to Production File

| What you learned | Lesson | Production file |
|---|---|---|
| How LLMs work, prompts, models | 05-07 | (Conceptual — informs all decisions) |
| Agent creation (name, model, instructions, tools) | 08-09 | `output/agents/researcher.py`, `output/agents/outliner.py`, `output/agents/writer.py`, `output/agents/image.py` |
| Pydantic schemas (ContentOutline, EnrichedContent) | 10, 13 | `output/agents/schemas.py` |
| Agent chaining, mini pipeline, full pipeline | 11-12, 13-15 | `output/pipeline.py` |
| Image toolkits (FreepikTools, DataForSEOTools) | 09, 14 | `output/agents/image.py` |
| Database (Airtable, CRUD functions) | 16 | `output/tools/airtable.py` |
| How everything connects | 17 | All files — the full request flow |
| Agno Team (leader + members + workspace tools) | 18 | `output/chat.py` + `output/tools/workspace.py` |

### File structure

```
output/
├── chat.py                     # Chat interface — THE entry point (lesson 18)
├── pipeline.py                 # The 4-step pipeline (lesson 15)
├── agents/                     # All agents, schemas, and image toolkits (lessons 8-14)
│   ├── __init__.py             # Re-exports all agents and schemas
│   ├── schemas.py              # Pydantic schemas (ContentOutline, EnrichedContent, etc.)
│   ├── researcher.py           # Research agent (Claude Sonnet + DuckDuckGo)
│   ├── outliner.py             # Outline agent (Claude Sonnet + output_schema)
│   ├── writer.py               # Writer agent (Grok-4, plain Markdown)
│   ├── image.py                # Image agent + FreepikTools/DataForSEOTools toolkits
│   ├── content_creator.py      # Content Creator team member
│   ├── status_tracker.py       # Status Tracker team member
│   ├── seo_manager.py          # SEO Manager team member
│   └── team.py                 # Agno Team definition (leader + members)
└── tools/                      # External APIs + integrations
    ├── __init__.py             # Re-exports
    ├── airtable.py             # Airtable database layer (lesson 16)
    ├── workspace.py            # Team member tools (lesson 18)
    └── rankings.py             # SERP rank checking via DataForSEO
```

**Tip:** When you want to modify something, use this table to find the right file. To change the writer's instructions, open `output/agents/writer.py` and edit the `writer_agent` instructions list.

## Exercise

Call the workspace tools directly (without the chat team) to explore the workspace:

1. Use `get_article_details(article_id)` to get details for an article (pick an ID from the list above)
2. Parse the JSON result and print the topic, status, and word count
3. If the article has content, use `get_article_content(article_id)` to read the first 500 characters

This is what the Status Tracker agent does when you ask "What's the status of article recABC123?" — you're calling the same functions.

In [ ]:
# Exercise: Fill in the blanks to explore the workspace

import json

# 1. Get article details (pick an ID from the list above)
article_id = "recXXX"  # Change this to a real Airtable record ID
result = ___(article_id)

# 2. Parse the JSON result into a Python dict
article = json.___(result)

# 3. Print the topic, status, and word count
print(f"Topic: {article[___]}")
print(f"Status: {article[___]}")
print(f"Word count: {article.get(___, 'N/A')}")

# 4. If the article has content, read the first 500 characters
content = ___(article_id)
if content and not content.startswith("Article"):
    print(f"\nFirst 500 chars:\n{content[:___]}")

<details>
<summary>Click to reveal answer</summary>

```python
import json

# 1. Get article details (pick an ID from the list above)
article_id = "recXXX"  # Change this to a real Airtable record ID
result = get_article_details(article_id)

# 2. Parse the JSON result into a Python dict
article = json.loads(result)

# 3. Print the topic, status, and word count
print(f"Topic: {article['topic']}")
print(f"Status: {article['status']}")
print(f"Word count: {article.get('word_count', 'N/A')}")

# 4. If the article has content, read the first 500 characters
content = get_article_content(article_id)
if content and not content.startswith("Article"):
    print(f"\nFirst 500 chars:\n{content[:500]}")
```

**Note:** `get_article_details()` returns JSON (so we use `json.loads()`), but `get_article_content()` returns plain Markdown text directly — no JSON parsing needed.
</details>

## Module 5 Complete

You now understand the **complete product** — from Airtable storage to architecture to the conversational AI team.

What you've covered:

### Module 1: Python Basics (Lessons 1-4)
Variables, lists, dictionaries, functions, packages — the foundation.

### Module 2: Understanding AI (Lessons 5-7)
How LLMs work, prompts and context, model choices — the "why" behind everything.

### Module 3: Building Agents (Lessons 8-12)
First agent, tools, structured output, chaining, mini pipeline — hands-on building.

### Module 4: The Real SEO Pipeline (Lessons 13-15)
Research, outline, writer, image agents — the production pipeline.

### Module 5: The Complete Product (Lessons 16-18)
Airtable database, how everything connects, chat interface — everything tied together.

---

### Next: Module 6 — AI-Assisted Development

You understand the full system. But the goal was never to make you write code from scratch. In the final module, you'll learn how to use **Claude Code** — an AI assistant that reads your codebase and makes changes for you. You'll learn to:
- **Direct AI to build features** instead of coding them yourself
- **Verify changes** using everything you've learned in Modules 1-5
- **Extend the product** with new agents, tools, and capabilities

This is the real payoff: understanding the system well enough to guide AI to improve it.